In [ ]:
# Эксперименты только с mandl
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval
from tqdm import tqdm
# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

model_weights_path = "../TNDP_learning/output/inductive_random_graphs.pt"

mandl_experiments = [
    ("mumford1", 0.0, 0.5, 0.0),
    ("mumford1", 0.2, 0.5, 0.0),
    ("mumford1", 0.4, 0.5, 0.0),
    ("mumford1", 0.6, 0.5, 0.0),
    ("mumford1", 0.8, 0.5, 0.0),
    ("mumford1", 1.0, 0.5, 0.0),
]


# CSV файл
results_file = Path("var_pass_coef.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "dataset", "demand_time", "route_time", "connectivity", "seed",
            "ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов только main_eval
for dataset_name, dt, rt, ct in tqdm(mandl_experiments, desc="Evaluating mandl"):
    for seed in range (0, 10):
        try:
            run_name = f"mandl_eval_pp_{dt}_op_{rt}_cp_{ct}"

            with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
                cfg_eval = compose(
                    config_name="eval_model_mumford",
                    overrides=[
                        f"+eval={dataset_name}",
                        f"++eval.n_routes=5",
                        f"+model.weights={model_weights_path}",
                        f"++run_name={run_name}",
                        f"++experiment.seed={seed}",
                        f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                        f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                        f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    ]
                )

            metrics, unserved_demand = main_eval(cfg_eval)
            keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
            row = [dataset_name, dt, rt, ct, seed] + [round(metrics[k].item(), 4) for k in keys_order]

            with open(results_file, mode='a', newline='') as f:
                csv.writer(f).writerow(row)

        except Exception as e:
            print(f"[✗] Failed eval for {run_name}: {e}")